# Shared code for the SVR notebooks

Registry, feature sets, and scoring helpers shared by `3.2a PredictionSVM_Standard.ipynb` (direct
target) and `3.2b PredictionSVM_Residual.ipynb` (residual-on-baseline target). Extracted here because
a cell-by-cell diff between the two notebooks showed these pieces were byte-identical or functionally
identical (only docstrings differed) — kept in two places, a fix like the residual notebook's
`SVR_MAX_ITER` cap only landing in one of them was exactly the kind of drift this notebook removes.

The two notebooks' actual training logic (`GridSearchCV` in `3.2a` vs. a manual K-fold loop in `3.2b`,
forced by `3.2b`'s need to reconstruct counts from a residual prediction using `base_demand`) is
**not** here — it is fundamentally different between the two and stays in each notebook as explicit,
separate code.

Both notebooks pull this in with IPython's `%run "3.2 Prediction_CommonGround.ipynb"` — it
executes every code cell below directly into the caller's namespace (the same effect as
`from module import *`), so no separate `.py` file is needed and this stays a notebook like everything
else in the pipeline. It is never registered in `report.qmd` and has no `#| label:` cells: it is not
itself a report figure/table, only a dependency of the two that are.

In [1]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "Total_Trip_Start"
RANDOM_STATE = 42
SPLIT_DIR = Path("..") / "data" / "prediction_split"

# Training budget. SVR is O(n^2) to fit and O(n_SV * n) to predict, so the raw splits (up to 6.2M
# training rows) are far out of reach. We fit on a small balanced sample and select on a validation
# subsample; only the winning model per level is re-scored on the full validation split. The test
# split is never touched here -- it is used once, in 3.4, for the SVM-vs-NN comparison. Identical
# across both notebooks so their results stay directly comparable.
N_SCAN = 4_000       # per class, for the kernel scan  -> 8k rows
N_GRID = 8_000       # per class, for the grid search  -> 16k rows
N_VAL_SUBSAMPLE = 50_000

KERNELS = ["linear", "poly", "rbf"]

## The six levels

Three spatial levels (H3 res-7, census tract, community area) crossed with two temporal
levels (hourly, daily). Each split lives at
`data/prediction_split/<spatial>/<temporal>/{train,val,test}.parquet`; the spatial level determines
the name of the key column.

In [2]:
SPATIAL_LEVELS = {
    "h3_7":      "h3_index_7",
    "census":    "geoid10",
    "community": "commarea",
}
TEMPORAL_LEVELS = ["hourly", "daily"]

LEVELS = list(itertools.product(SPATIAL_LEVELS, TEMPORAL_LEVELS))


def load_splits(spatial, temporal):
    """Load train/val/test for one (spatial, temporal) level."""
    base = SPLIT_DIR / spatial / temporal
    return tuple(pd.read_parquet(base / f"{s}.parquet") for s in ("train", "val", "test"))

## Feature sets

Four candidate feature sets per level: a `basic` calendar + location + history block, then POI
counts and weather layered on top. Daily grids have no hour features — a daily row is floored to
midnight, so `hour_sin`/`hour_cos` would be constants — so the feature set is derived from the
temporal level rather than being a single global list.

In [3]:
HOUR = ["hour_sin", "hour_cos"]                    # hourly grids only
CALENDAR = ["month_sin", "month_cos", "is_weekend",
            "is_holiday", "is_near_holiday", "day_of_week"]
LOCATION = ["lat", "lon", "distance_to_loop"]
HISTORY = ["base_demand"]                          # train-only historical mean, computed in 3.1

POI_COLS = ["poi_cat_automotive", "poi_cat_civic_community", "poi_cat_education",
            "poi_cat_entertainment", "poi_cat_finance", "poi_cat_food_drink", "poi_cat_grocery",
            "poi_cat_health", "poi_cat_leisure_sports", "poi_cat_lodging", "poi_cat_nightlife",
            "poi_cat_services", "poi_cat_shopping", "poi_cat_transport"]
WEATHER = ["2m_temp_c", "total_precip_mm", "snow_cov", "snow_depth", "wind_speed"]


def feature_sets(temporal):
    """The four candidate feature sets for a given temporal level."""
    basic = (HOUR if temporal == "hourly" else []) + CALENDAR + LOCATION + HISTORY
    return {
        "basic": basic,
        "basic+poi": basic + POI_COLS,
        "basic+weather": basic + WEATHER,
        "basic+poi+weather": basic + POI_COLS + WEATHER,
    }


## Scoring helpers

- **log1p target.** Demand is counts with a long right tail. We fit on `log1p(y)` and invert with
  `expm1`, clipped at zero.
- **Balanced sample.** At h3_7/hourly the target is about 94% zeros; a uniform sample would be almost
  all zeros and the SVR would learn to predict nothing. We take all non-zero rows up to a cap plus an
  equal number of zeros. Where a level is *not* zero-inflated (`community/daily` is only 0.6% zeros)
  the zero class simply runs out and the sample is non-zero-heavy — that is the correct behaviour.
- **Metrics in count space**, not log space, so MAE is "trips per cell-period".
- **Skill score** is scale-free so it is comparable across levels — raw MAE is not, since mean demand
  ranges from about 2.5 per cell-hour at h3_7/hourly to over 200 per area-day at community/daily, an
  ~80x gap.

In [4]:
LOG_CEILING = 20.0   # expm1(20) ~ 4.9e8 pickups in one cell-period: already absurd


def to_counts(pred_log):
    """Invert log1p and clip negative demand to 0.

    The log prediction is clipped first. An unbounded linear kernel can predict a log-demand of
    several hundred, and expm1 of that is +inf -- which then propagates into r2_score as a crash
    rather than a bad score. Clipping keeps the linear kernel's failure *finite and reportable*
    without touching any prediction a sane model would make.
    """
    return np.clip(np.expm1(np.clip(pred_log, -LOG_CEILING, LOG_CEILING)), 0, None)


def score(y_true, pred_counts, name):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, pred_counts),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred_counts)),
        "R2": r2_score(y_true, pred_counts),
    }


def balanced_sample(df, n_per_class, seed=RANDOM_STATE):
    """All non-zero rows (capped) plus an equal number of zero rows, shuffled.

    Where a level is not zero-inflated the zero class runs out and min() caps it -- the sample is then
    simply non-zero-heavy, which is what we want.
    """
    nz = df[df[TARGET] > 0]
    z = df[df[TARGET] == 0]
    return pd.concat([
        nz.sample(n=min(n_per_class, len(nz)), random_state=seed),
        z.sample(n=min(n_per_class, len(z)), random_state=seed),
    ]).sample(frac=1, random_state=seed)


def skill(model_metrics, baseline_metrics):
    """Fraction of the baseline's error the model removes. 0 = no better than baseline, 1 = perfect.

    Raw MAE cannot be compared across levels -- mean demand is about 2.5 per cell-hour at h3_7/hourly
    but over 200 per area-day at community/daily, an ~80x scale gap, so the finest grid would always
    'win' on MAE simply by having less to be wrong about. Skill is scale-free: it asks whether the SVR
    beats the historical-mean baseline ON ITS OWN GROUND, which is comparable between levels.
    """
    return {
        "skill_MAE": 1 - model_metrics["MAE"] / baseline_metrics["MAE"],
        "skill_RMSE": 1 - model_metrics["RMSE"] / baseline_metrics["RMSE"],
    }

## Shared tables and plots

Cells that were byte-/functionally-identical between `3.2a` and `3.2b` — the per-level training
overview, the feature-set-size print, the winning-model comparison table, and the skill-by-level
bar chart — live here as helpers so both notebooks render them the same way and cannot drift. Each
caller keeps only its own `#| label:` / `tbl-cap` / `fig-cap` cell that invokes the helper.

The `plot_skill_by_level` palette (`SERIES`, `INK`, `MUTED`, `GRID`) is defined at module scope
because the feature-importance heatmap in `3.2a` reuses the same ink colours.

In [ ]:
# Shared plot palette (skill chart + the 3.2a feature-importance heatmap)
SERIES = {"hourly": "#2a78d6", "daily": "#1baf7a"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#d8d7d2"


def overview_table():
    """Per-level training-set overview: units, rows, mean demand, zero share.

    The zero share differs a lot between levels, which matters when interpreting the results.
    """
    rows = []
    for spatial, temporal in LEVELS:
        df = pd.read_parquet(SPLIT_DIR / spatial / temporal / "train.parquet",
                             columns=[SPATIAL_LEVELS[spatial], TARGET])
        rows.append({
            "spatial": spatial, "temporal": temporal,
            "units": df[SPATIAL_LEVELS[spatial]].nunique(),
            "train rows": len(df),
            "mean demand": df[TARGET].mean(),
            "zero %": (df[TARGET] == 0).mean() * 100,
        })
        del df
    return pd.DataFrame(rows).round(2)


def print_feature_set_sizes():
    """Print the four feature-set sizes for each temporal level."""
    for temporal in TEMPORAL_LEVELS:
        sizes = {name: len(f) for name, f in feature_sets(temporal).items()}
        print(f"{temporal:7s} {sizes}")


def build_level_comparison(bundles, out_csv=None):
    """Assemble the per-level winning-model comparison table, sorted by skill_MAE.

    `bundles` maps (spatial, temporal, feature_set) -> bundle; the winner of each level carries
    is_level_winner=True plus the validation/skill metrics attached during training (scored on the
    full validation split -- the test split is reserved for 3.4). Optionally writes the table to
    `out_csv`.
    """
    comparison = []
    for spatial, temporal in LEVELS:
        best = next(b for k, b in bundles.items()
                    if k[0] == spatial and k[1] == temporal and b.get("is_level_winner"))
        comparison.append({
            "spatial": spatial,
            "temporal": temporal,
            "feature_set": best["feature_set"],
            "kernel": best["kernel"],
            #"C": best["best_params"].get("C"),
            #"gamma": best["best_params"].get("gamma"),
            "base_MAE": best["baseline_val_metrics"]["MAE"],
            "val_MAE": best["val_full_metrics"]["MAE"],
            "val_RMSE": best["val_full_metrics"]["RMSE"],
            "val_R2": best["val_full_metrics"]["R2"],
            "skill_MAE": best["skill"]["skill_MAE"],
            "skill_RMSE": best["skill"]["skill_RMSE"],
        })
    level_comparison = pd.DataFrame(comparison).sort_values("skill_MAE", ascending=False)
    if out_csv is not None:
        level_comparison.to_csv(out_csv, index=False)
    return level_comparison


def plot_skill_by_level(level_comparison, title, temporal_levels=TEMPORAL_LEVELS):
    """Grouped vertical bar chart of SVR skill vs. the historical-mean baseline, per spatial level and
    temporal granularity. The one standardized skill chart shared by 3.2a (direct) and 3.2b
    (residual-on-baseline). Spatial levels are ordered coarse -> detailed, filtered to those present,
    so it adapts automatically to the registry in SPATIAL_LEVELS."""
    coarse_to_fine = [s for s in ["community", "census", "h3_7"]
                      if s in set(level_comparison["spatial"])]
    # Reindex both axes so a missing spatial or temporal level shows as an empty bar rather than
    # raising -- keeps the chart robust to a partial level_comparison.
    skill_grid = (level_comparison
                  .pivot(index="spatial", columns="temporal", values="skill_MAE")
                  .reindex(index=coarse_to_fine, columns=temporal_levels))

    x = np.arange(len(coarse_to_fine))
    width = 0.38
    fig, ax = plt.subplots(figsize=(8, 4.5))

    for i, temporal in enumerate(temporal_levels):
        vals = skill_grid[temporal].to_numpy()
        pos = x + (i - 0.5) * width
        ax.bar(pos, vals, width * 0.94, label=temporal, color=SERIES[temporal], zorder=3)
        for p, v in zip(pos, vals):
            if np.isnan(v):
                continue
            ax.annotate(f"{v:+.1%}", (p, v), textcoords="offset points",
                        xytext=(0, 4 if v >= 0 else -13), ha="center",
                        fontsize=9, color=INK)

    ax.axhline(0, color=MUTED, lw=1.2, zorder=4)
    ax.set_xticks(x, [o.replace("_", " res-") for o in coarse_to_fine])
    ax.set_ylabel("Skill vs. historical-mean baseline (MAE)")
    ax.set_xlabel("Spatial granularity (coarse → detailed)")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.yaxis.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    ax.legend(frameon=False, loc="best", labelcolor=INK)
    ax.set_title(title, color=INK, fontsize=12, loc="left", pad=12)
    plt.tight_layout()
    plt.show()